# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method Choice and Why

For this assignment, I chose Logistic Regression because my prediction task is a simple yes/no classification problem. I wanted to predict whether a page's CTR is below the expected median for its position tier.

In Week 4, I built a rule-based baseline using page freshness and CTR. That worked well for ranking pages, but it depended directly on CTR, so it couldn't explain which other factors might be related to underperforming pages.

This week, I wanted to see if page and content information could predict underperformance without relying on CTR itself. To avoid data leakage, I left out features like CTR, clicks, sessions, pageviews, engagement rate, scroll rate, AI traffic percentage, and trend-related columns because they are directly connected to the outcome I'm trying to predict.

I kept features such as average position, position tier, word count, content type, search intent, competition, freshness, and the AI model used. These features provide useful information while keeping the model fair. I also removed rows where `avg_position` was 0 because those pages have no ranking data.

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target-related columns
df["has_position"] = df["avg_position"] > 0
df["is_stale"] = df["days_since_last_update"] >= 180
df["is_visible"] = df["impressions_90d"] >= 500

median_ctr_per_tier = (
    df[df["has_position"]]
    .groupby("position_tier")["ctr"]
    .median()
)

df["expected_ctr"] = df["position_tier"].map(median_ctr_per_tier)

df["ctr_underperforming"] = (
    df["has_position"] &
    (df["ctr"] < df["expected_ctr"])
)

print("Rows:", len(df))
print("Rows with position:", df["has_position"].sum())
print(
    "Positive rate:",
    round(df.loc[df["has_position"], "ctr_underperforming"].mean(), 3)
)

Rows: 30000
Rows with position: 28795
Positive rate: 0.454


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split Design

Instead of using a random split, I grouped the data by client. This prevents pages from the same client appearing in both the training and testing sets, which could make the model learn client-specific patterns instead of general ones.

Since this dataset is only a single 90-day snapshot, a time-based split wasn't possible.

I used StratifiedGroupKFold with 5 folds because it keeps clients together while maintaining a similar balance of positive and negative examples in every fold. This gives a more reliable evaluation, and every page is tested using a model that never saw data from the same client during training.

In [9]:
model_df = df[df["has_position"]].copy()

model_df["provider_used"] = model_df["provider_used"].fillna("unknown")

for col in [
    "word_count",
    "char_count",
    "search_volume",
    "competition",
    "cpc",
]:
    model_df[f"has_{col}"] = model_df[col].notna().astype(int)

# Target
y = model_df["ctr_underperforming"].astype(int)

# Client groups
groups = model_df["client_id"]

# Numerical features
feature_cols_num = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "avg_position",
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "days_since_last_update",
    "content_age_days",
    "days_with_impressions",
    "has_word_count",
    "has_char_count",
    "has_search_volume",
    "has_competition",
    "has_cpc",
]

# Categorical features
feature_cols_cat = [
    "content_type",
    "main_intent",
    "competition_level",
    "position_tier",
    "provider_used",
    "model_used",
]

X = model_df[feature_cols_num + feature_cols_cat]

print("Rows modeled:", len(model_df))
print("Clients:", groups.nunique())
print("Positive rate:", round(y.mean(), 3))

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):
    print(
        f"Fold {fold + 1}: "
        f"Train={len(train_idx)}, "
        f"Test={len(test_idx)}"
    )

Rows modeled: 28795
Clients: 31
Positive rate: 0.454
Fold 1: Train=23906, Test=4889
Fold 2: Train=23560, Test=5235
Fold 3: Train=16553, Test=12242
Fold 4: Train=22860, Test=5935
Fold 5: Train=28301, Test=494


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + Compare vs My Baseline

I couldn't compare the model directly with my Week 4 rule because that rule already uses CTR to identify underperforming pages, and CTR is also the target I'm trying to predict. Using it would leak the answer into the model.

Instead, I created a simple position-based baseline. It predicts a page as underperforming if its average position is worse than the median position of its tier.

Both the baseline and Logistic Regression were evaluated using the same grouped cross-validation so the comparison is fair. I compared the models using Accuracy, Precision, Recall, F1-score, and ROC-AUC.

In [10]:
tier_median_position = (
    model_df.groupby("position_tier")["avg_position"]
    .transform("median")
)

baseline_pred = (
    model_df["avg_position"] > tier_median_position
).astype(int)

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, feature_cols_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
])

# Logistic Regression
clf = Pipeline([
    ("prep", preprocess),
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    )),
])

oof_pred = np.zeros(len(X), dtype=int)
oof_proba = np.zeros(len(X))

for train_idx, test_idx in sgkf.split(X, y, groups):

    clf.fit(X.iloc[train_idx], y.iloc[train_idx])

    probability = clf.predict_proba(X.iloc[test_idx])[:, 1]

    oof_proba[test_idx] = probability

    oof_pred[test_idx] = (probability >= 0.5).astype(int)

# Evaluation function
def evaluate(name, y_true, pred, proba=None):
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_true, pred), 3),
        "Precision": round(precision_score(y_true, pred), 3),
        "Recall": round(recall_score(y_true, pred), 3),
        "F1": round(f1_score(y_true, pred), 3),
        "ROC_AUC": round(roc_auc_score(y_true, proba), 3)
        if proba is not None else None,
    }

comparison = pd.DataFrame([
    evaluate("Position Baseline", y, baseline_pred),
    evaluate("Logistic Regression", y, oof_pred, oof_proba),
])

comparison

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Position Baseline,0.547,0.502,0.544,0.522,NaN
1,Logistic Regression,0.731,0.752,0.610,0.674,0.796


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and Interpretation

The Logistic Regression model performed much better than the simple position-based baseline. It achieved an accuracy of 73.1% and an F1-score of 0.674, showing that the selected features contain useful information for predicting CTR underperformance.

The confusion matrix shows that the model correctly classified most pages, although it still made some false positives and false negatives. This is expected because some pages have very similar characteristics and are difficult to separate.

The permutation importance results show that **days_with_impressions**, **position_tier**, **model_used**, and **avg_position** had the biggest influence on the predictions. These features appear to provide the strongest signal for identifying underperforming pages.

Overall, I see this model as a decision-support tool rather than something that should make decisions automatically. It can help identify pages that deserve further review, but the final decision should still involve human judgment.

In [11]:
model_df["prediction"] = oof_pred

model_df["correct"] = (
    model_df["prediction"] == y
).astype(int)

print("Overall Accuracy:", round(model_df["correct"].mean(), 3))

print("\nConfusion Matrix\n")

print(
    pd.DataFrame(
        confusion_matrix(y, oof_pred),
        index=[
            "Actual Negative",
            "Actual Positive",
        ],
        columns=[
            "Predicted Negative",
            "Predicted Positive",
        ],
    )
)

# Train final model
clf.fit(X, y)

importance = permutation_importance(
    clf,
    X,
    y,
    scoring="f1",
    random_state=RANDOM_SEED,
    n_repeats=10,
)

importance = (
    pd.Series(
        importance.importances_mean,
        index=X.columns,
    )
    .sort_values(ascending=False)
)

print("\nTop 10 Important Features\n")
print(importance.head(10))

Overall Accuracy: 0.731

Confusion Matrix

                 Predicted Negative  Predicted Positive
Actual Negative               13078                2638
Actual Positive                5098                7981

Top 10 Important Features

days_with_impressions    0.179379
position_tier            0.092321
model_used               0.057607
avg_position             0.040188
competition_level        0.036636
main_intent              0.014508
content_type             0.007076
impressions_last_30d     0.006471
has_cpc                  0.005667
has_competition          0.005667
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.